# 통신 채널로서의 잔차 스트림 - 잔차 스트림과 부분 공간

- Tutorial ID: `tut-5-1`
- Tutorial: 통신 채널로서의 잔차 스트림
- Section ID: `s5-1-1`
- Section: 잔차 스트림과 부분 공간


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
from scipy import linalg

print("=" * 60)
print("잔차 스트림 분석: 부분 공간과 대역폭")
print("=" * 60)

np.random.seed(42)

d_model = 16  # 잔차 스트림 차원
d_head = 4    # 어텐션 헤드 차원
n_heads = 4   # 헤드 수

In [ ]:
# 1. 부분 공간 분리 시뮬레이션

In [ ]:
print("
1. 부분 공간 분리 시뮬레이션")
print("-" * 40)

# 각 헤드가 서로 다른 부분 공간에서 작동한다고 가정
# 실제 트랜스포머에서는 훈련으로 학습됨

# 헤드별 출력 공간 (각 헤드는 d_head 차원 부분 공간에 씀)
head_outputs = []
for h in range(n_heads):
    # 헤드 h는 주로 차원 [h*4 : (h+1)*4]에 씀
        output = np.zeros(d_model)
    output[h*d_head : (h+1)*d_head] = np.random.randn(d_head)
    head_outputs.append(output)

# 잔차 스트림 = 모든 헤드 출력의 합
residual = np.zeros(d_model)
for h, ho in enumerate(head_outputs):
    residual += ho
    print(f"  헤드 {h} 출력 (활성 차원 {h*d_head}-{(h+1)*d_head-1}): "
          f"norm={np.linalg.norm(ho):.3f}")

print(f"
  잔차 스트림 총 norm: {np.linalg.norm(residual):.3f}")
print(f"  헤드들이 직교 부분 공간 사용 → 간섭 최소화")

In [ ]:
# 2. 특권 기저 없음 (No Privileged Basis)

In [ ]:
print("
2. 특권 기저 없음 (회전 불변성)")
print("-" * 40)

# 임의 회전 행렬 생성 (직교 행렬)
Q, _ = np.linalg.qr(np.random.randn(d_model, d_model))
rotation = Q  # Q는 직교 행렬: Q^T Q = I

# 회전 전 행렬들
W_E = np.random.randn(8, d_model) * 0.3  # 임베딩
W_Q = np.random.randn(d_model, d_head) * 0.3  # 쿼리

# 잔차 스트림을 Q로 회전
# W_E_rot @ Q^T @ Q @ W_Q = W_E_rot @ W_Q (불변!)
W_E_rot = W_E @ rotation.T
W_Q_rot = rotation @ W_Q

# 결과가 동일함을 확인
result_original = W_E @ W_Q
result_rotated = W_E_rot @ W_Q_rot

print(f"  원래 W_E @ W_Q norm: {np.linalg.norm(result_original):.6f}")
print(f"  회전 후 W_E_rot @ W_Q_rot norm: {np.linalg.norm(result_rotated):.6f}")
print(f"  차이: {np.linalg.norm(result_original - result_rotated):.10f}")
print(f"  → 잔차 스트림을 임의 회전해도 모델 동작 불변!")

In [ ]:
# 3. 대역폭 경쟁 시뮬레이션

In [ ]:
print("
3. 잔차 스트림 대역폭 경쟁")
print("-" * 40)

# 50층 GPT-2 large 수준
d_model_large = 1280
n_layers = 36
mlp_expansion = 4
n_heads_large = 20

# 각 층의 계산 차원
attn_dims = n_heads_large * (d_model_large // n_heads_large)  # = d_model
mlp_dims = mlp_expansion * d_model_large

print(f"  모델: d_model={d_model_large}, n_layers={n_layers}")
print(f"  잔차 스트림 대역폭: {d_model_large} 차원")
print(f"  레이어당 어텐션 차원: {attn_dims}")
print(f"  레이어당 MLP 뉴런: {mlp_dims} ({mlp_expansion}×d_model)")
print(f"  전체 계산 차원: {n_layers * (attn_dims + mlp_dims):,}")
print(f"  비율 (계산/대역폭): {n_layers * (attn_dims + mlp_dims) / d_model_large:.1f}x")
print(f"
  → 잔차 스트림은 엄청난 압력을 받습니다!")
print(f"  → '중첩(Superposition)'으로 여러 정보를 동시 저장")

In [ ]:
# 4. PCA로 임베딩 구조 분석

In [ ]:
print("
4. PCA로 임베딩/언임베딩 구조 분석")
print("-" * 40)

vocab_size = 50
W_E_sim = np.random.randn(vocab_size, d_model)  # 임베딩
W_U_sim = np.random.randn(d_model, vocab_size)  # 언임베딩

# SVD (PCA와 동일)
U_e, S_e, Vt_e = np.linalg.svd(W_E_sim)
U_u, S_u, Vt_u = np.linalg.svd(W_U_sim.T)

print(f"  임베딩 W_E 특이값 스펙트럼:")
print(f"  {np.round(S_e[:8], 2)}")
print(f"  언임베딩 W_U 특이값 스펙트럼:")
print(f"  {np.round(S_u[:8], 2)}")
print(f"
  논문 발견: 실제 대형 모델에서 이 스펙트럼이 빠르게 감소")
print(f"  → 임베딩/언임베딩은 전체 d_model의 작은 부분만 사용")
print(f"  → 나머지 차원은 어텐션 헤드와 MLP가 자유롭게 사용!")